# Scrape Radio Radicale

Given a URL:
1. download audio
2. get timestamps and speech date
3. trim audio to timestamps
4. text to speech
5. save into csv 
   1. politician name
   2. date of speech
   3. metadata
      1. url
      2. others
   4. transcribed text

In [5]:
import os
from pathlib import Path
testing = True

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))  # go to ~/crossdem/ by jumping up twice
OUT_DIR = os.path.join(BASE_DIR, "datasets")
if testing:
    OUT_DIR = os.path.join(BASE_DIR, "notebooks/out")
os.makedirs(OUT_DIR, exist_ok=True)

#with open(os.path.join(OUT_DIR, "output.txt"), "w") as f:
#    f.write("hello world")
#
#print(f"Written to {OUT_DIR}/output.txt")

## Download audio
Given a radio radicale page url, extract the audio embedding url and downloads it. Also embed metadata in the mp3 file.

In [ ]:
import re
import json
import requests
import yt_dlp

METADATA_FIELDS = ("title", "description", "upload_date", "duration", "tags", "categories")

def get_audio_url(page_url: str) -> str:
    resp = requests.get(page_url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    match = re.search(r'rtsp://[^"]+\.mp3', resp.text)
    if not match:
        raise ValueError("No rtsp audio link found on the page")
    return match.group(0)

def download_audio(url: str, out_dir: str) -> dict:
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": f"{out_dir}/{url[-5:]}_%(title).10s.%(ext)s",        #~/notebooks/out/12345_abitoftitle.mp3
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            },
            {
                "key": "FFmpegMetadata",
                "add_metadata": True,
            },
        ],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.sanitize_info(info)



if __name__ == "__main__":
    PAGE_URL = "https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823"
    #OUT_DIR = "./"

    try:
        video_metadata = download_audio(PAGE_URL, OUT_DIR)
    except yt_dlp.utils.DownloadError:
        print("yt-dlp couldn't extract directly, falling back to scrape...")
        audio_url = get_audio_url(PAGE_URL)
        print("Found:", audio_url)
        video_metadata = download_audio(audio_url, OUT_DIR)


[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-s...onsabilita?i=1385823
[RadioRadicale] 157501: Downloading webpage
[RadioRadicale] 157501-0: Downloading m3u8 information
[info] 157501: Downloading 1 format(s): 63
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 422
[download] Destination: /home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .m4a
[download] 100% of   38.68MiB in 00:00:46 at 848.48KiB/s                 
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .m4a"
[ExtractAudio] Destination: /home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .mp3
Deleting original file /home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .m4a (pass -k to keep)
[Metadata] Adding metadata to "/home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .mp3"

--- Metadata --

All video metadata all extracted and embedded in the mp3 file 

In [ ]:
for field, value in video_metadata.items():
    print(f"{field:<20}: {value}")

id                  : 157501
title               : "La crisi del governo Fanfani e i referendum radicali su nucleare e responsabilità civile dei magistrati..." servizio realizzato con documentazione tratta dall'archivio di Radio Radicale
formats             : [{'format_id': '63', 'format_index': None, 'url': 'https://video.radioradicale.it/store-78/_definst_/mp3:mp004/MP266557.mp3/chunklist_w86837949.m3u8', 'manifest_url': 'https://video.radioradicale.it/store-78/_definst_/mp3:mp004/MP266557.mp3/playlist.m3u8', 'tbr': 63.699, 'ext': 'm4a', 'fps': None, 'protocol': 'm3u8_native', 'preference': None, 'quality': None, 'has_drm': False, 'vcodec': 'none', 'acodec': 'mp4a.40.34', 'dynamic_range': None, 'audio_ext': 'm4a', 'video_ext': 'none', 'vbr': 0, 'abr': 63.699, 'resolution': 'audio only', 'aspect_ratio': None, 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36', 'Accept': 'text/html,application

## Extract speech date & timestamps
Uses BeautifulSup to parse all XML data and try to find out when a speech was made

In [43]:
import re
from datetime import date

import requests
from bs4 import BeautifulSoup


# ── HTTP ────────────────────────────────────────────────────────────────────

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "it-IT,it;q=0.9,en;q=0.5",
}

def fetch_page(url: str, timeout: int = 20) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ── Date patterns ───────────────────────────────────────────────────────────

MONTHS_IT = {
    "gennaio": 1, "febbraio": 2, "marzo": 3, "aprile": 4,
    "maggio": 5, "giugno": 6, "luglio": 7, "agosto": 8,
    "settembre": 9, "ottobre": 10, "novembre": 11, "dicembre": 12,
}
MONTHS_IT_SHORT = {
    "gen": 1, "feb": 2, "mar": 3, "apr": 4, "mag": 5, "giu": 6,
    "lug": 7, "ago": 8, "set": 9, "ott": 10, "nov": 11, "dic": 12,
}

RE_DATE_LONG   = re.compile(r"\b(\d{1,2})\s+(" + "|".join(MONTHS_IT) + r")\s+(\d{4})\b", re.I)
RE_DATE_SHORT  = re.compile(r"\b(\d{1,2})\s+(" + "|".join(MONTHS_IT_SHORT) + r")\s+(\d{4})\b", re.I)
RE_DATE_DOTTED = re.compile(r"\b(\d{1,2})\.(\d{2})\.(\d{4})\b")
RE_DATE_ISO    = re.compile(r"(\d{4})-(\d{2})-(\d{2})")

def _try_long(text):   m = RE_DATE_LONG.search(text);   return date(int(m[3]), MONTHS_IT[m[2].lower()],       int(m[1])) if m else None
def _try_short(text):  m = RE_DATE_SHORT.search(text);  return date(int(m[3]), MONTHS_IT_SHORT[m[2].lower()], int(m[1])) if m else None
def _try_dotted(text): m = RE_DATE_DOTTED.search(text); return date(int(m[3]), int(m[2]),                      int(m[1])) if m else None
def _try_iso(text):    m = RE_DATE_ISO.search(text);    return date(int(m[1]), int(m[2]),                      int(m[3])) if m else None

def find_date(text: str) -> date | None:
    return _try_long(text) or _try_short(text) or _try_dotted(text) or _try_iso(text)


# ── Timestamp parsing ───────────────────────────────────────────────────────

RE_DURATA = re.compile(r'(\d+:\d+)\s+Durata:\s+(\d+)\s+min', re.I)

def parse_timestamps(li) -> dict:
    """
    Extract timing info from a speaker <li>.
    Returns a dict with:
      start_time    – e.g. '0:35'  (mm:ss within the recording)
      duration_min  – e.g. 45
      start_seconds – e.g. 2700  (from the d<N> CSS class, redundant but exact)
    All values are None if the durata div is absent or unparseable.
    """
    result = {"start_time": None, "duration_min": None, "start_seconds": None}

    durata_div = li.find("div", class_="durata")
    if durata_div:
        m = RE_DURATA.search(durata_div.get_text())
        if m:
            result["start_time"]   = m.group(1)
            result["duration_min"] = int(m.group(2))

    # d<seconds> class is always present and is the most machine-readable form
    d_class = next(
        (c for c in li.get("class", []) if re.match(r"d\d+$", c)),
        None,
    )
    if d_class:
        result["start_seconds"] = int(d_class[1:])

    return result


# ── Date extraction ─────────────────────────────────────────────────────────

def extract_page_date(soup: BeautifulSoup) -> tuple[date | None, str]:
    """Page-level event date (meta > title > stamp > sommario prose)."""
    for attr, val in [("name", "dcterms.date"), ("property", "article:published_time")]:
        tag = soup.find("meta", {attr: val})
        if tag and tag.get("content"):
            d = _try_iso(tag["content"])
            if d: return d, "meta_tag"

    title = soup.find("title")
    if title:
        d = _try_dotted(title.get_text())
        if d: return d, "page_title"

    page_text = soup.get_text(" ", strip=True)
    d = _try_short(page_text)
    if d: return d, "page_stamp"

    d = _try_long(page_text)
    if d: return d, "page_sommario"

    return None, "not_found"


def extract_speaker_date(soup: BeautifulSoup, speaker: str) -> tuple[date | None, str]:
    """
    Per-speaker date, searched in two places:
      1. The speaker's own int_subtext
      2. int_subtext of the *preceding* li (e.g. 'L\'on. Fanfani a una
         conferenza stampa del 22 marzo 1962' appears under a different speaker
         introducing the clip)
    """
    items = [li for li in soup.select("li.intervento") if li.find("h2")]

    for idx, li in enumerate(items):
        if speaker.lower() not in li.find("h2").get_text().lower():
            continue

        # Check the speaker's own subtext first
        subtext = li.find("div", class_="int_subtext")
        if subtext:
            d = find_date(subtext.get_text())
            if d: return d, "speaker_subtext"

        # Check the preceding li's subtext (introduction pattern)
        if idx > 0:
            prev_subtext = items[idx - 1].find("div", class_="int_subtext")
            if prev_subtext:
                text = prev_subtext.get_text()
                if speaker.lower() in text.lower():
                    d = find_date(text)
                    if d: return d, "preceding_subtext"

        # Fallback: any date anywhere in the li block
        d = find_date(li.get_text(" ", strip=True))
        if d: return d, "speaker_block"

        return None, "speaker_block_no_date"

    return None, "speaker_not_found"


# ── Main ────────────────────────────────────────────────────────────────────

def extract_date_and_timestamps(url: str, speaker: str = "Fanfani", timeout: int = 20) -> dict:
    """
    Fetch a Radio Radicale scheda page and return:
      speech_date     – ISO date of the speaker's actual speech, or None
      date_source     – how the date was found
      page_event_date – ISO date of the page-level event
      start_time      – 'mm:ss' offset in the recording, or None
      duration_seconds   – same as integer seconds (from d<N> CSS class), or None
      duration_min    – duration in minutes, or None
      speaker_found   – bool
      error           – error string or None
    """
    result = dict(
        url=url, speaker_query=speaker, speaker_found=False,
        speech_date=None, date_source=None, page_event_date=None,
        start_time=None, duration_seconds=None, duration_min=None,
        error=None,
    )
    try:
        soup = fetch_page(url, timeout=timeout)
    except Exception as e:
        result["error"] = str(e)
        return result

    page_date, page_src = extract_page_date(soup)
    if page_date:
        result["page_event_date"] = page_date.isoformat()

    # Find the speaker's <li> and extract timestamps
    for li in soup.select("li.intervento"):
        h2 = li.find("h2")
        if h2 and speaker.lower() in h2.get_text().lower():
            result["speaker_found"] = True
            ts = parse_timestamps(li)
            result["start_time"]    = ts["start_time"]    or "no timestamps"
            result["duration_seconds"] = ts["start_seconds"]
            result["duration_min"]  = ts["duration_min"]
            break

    if not result["speaker_found"]:
        result["error"] = f"'{speaker}' not found in interventi"
        result["date_source"] = "llm_needed"
        return result

    spk_date, spk_src = extract_speaker_date(soup, speaker)

    if spk_date:
        result["speech_date"] = spk_date.isoformat()
        result["date_source"] = spk_src
    elif page_date:
        result["speech_date"] = page_date.isoformat()
        result["date_source"] = f"page_event ({page_src})"
    else:
        result["date_source"] = "llm_needed"

    return result

In [46]:
# ── Run ─────────────────────────────────────────────────────────────────────
urls = [
    "https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823",
    "https://www.radioradicale.it/scheda/3913/crisi-di-governo-le-consultazioni-di-fanfani?i=2809940",
    "https://www.radioradicale.it/scheda/115568/le-elezioni-anticipate-della-primavera-87-il-governo-fanfani-lopposizione-radicale?i=1806617",
    "https://www.radioradicale.it/scheda/20978/crisi-di-governo-il-presidente-incaricato?i=2725312",
    "https://www.radioradicale.it/scheda/58768/sindacato-crisi-della-sua-rappresentanza-e-sfida-del-cambiamento?i=2419711"
]

for url in urls:
    r = extract_date_and_timestamps(url, speaker="Fanfani")
    for key, value in r.items():
        print(f"{key:<18} {value}")

url                https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823
speaker_query      Fanfani
speaker_found      True
speech_date        1987-04-21
date_source        speaker_subtext
page_event_date    2004-09-03
start_time         0:00
duration_seconds   1560
duration_min       26
error              None
url                https://www.radioradicale.it/scheda/3913/crisi-di-governo-le-consultazioni-di-fanfani?i=2809940
speaker_query      Fanfani
speaker_found      True
speech_date        1982-11-27
date_source        page_event (meta_tag)
page_event_date    1982-11-27
start_time         no timestamps
duration_seconds   1
duration_min       None
error              None
url                https://www.radioradicale.it/scheda/115568/le-elezioni-anticipate-della-primavera-87-il-governo-fanfani-lopposizione-radicale?i=1806617
speaker_query      Fanfani
speaker_found      True
speech_date        1987-04-20
d

## Cut audio timestamps TODO
implement into the pipeline

In [ ]:
import subprocess

# Your extracted data dictionary
r = {
    "start_time": "0:00",
    "duration_seconds": 1680
}

input_file = "downloaded_audio.mp3"
output_file = "trimmed_audio.mp3"

# FFmpeg command
# -ss defines the start time
# -t defines the duration in seconds
command = [
    "ffmpeg",
    "-y",               # Overwrite output file if it exists
    "-ss", r["start_time"],
    "-i", input_file,
    "-t", str(r["duration_seconds"]),
    "-acodec", "copy",  # 'copy' avoids re-encoding, making it instant
    output_file
]

# Run the command
subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
print("Audio sliced successfully via FFmpeg!")

## Text to speech 
Take trimmed speeches and transcribe them into text

In [ ]:
import json
import os
import subprocess
import csv
from datetime import datetime
from pathlib import Path

In [ ]:
politician  = "meloni"
model_name  = "medium"            # tiny, base, medium, large
lang_code   = "Italian"
# Use BASE_DIR to join paths
#audio_file  = str(BASE_DIR / "audio.mp3")

output_dir  = str(BASE_DIR / f"output-{model_name}")
csv_dir     = BASE_DIR / "csv_out"
audio_dir   = BASE_DIR / "audio_out"

csv_dir.mkdir(parents=True, exist_ok=True)

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
def process_video(video_url):
    unique_id = video_url[-5:]
    audio_file  = str(audio_dir / f"{unique_id}_audio.mp3")
    csv_output  = str(csv_dir / f"{unique_id}_meloni_speech2text.csv")

    # 1. Download metadata and audio in a single pass
    print(f"Fetching metadata and downloading audio of video {video_url}...")

    # Use --print-json to get metadata without creating a temporary .json file
    # Use -x and --audio-format to handle the extraction internally
    result = subprocess.run([
        "yt-dlp",
        "--print-json",
        "-x",                               # download audio only
        "--audio-format", "mp3",            
        "--cookies", "yt_cookies.txt",      # manual cookies, extract with browser extension
        "--sleep-requests", "2",            # Sleep 2s between requests
        "--sleep-interval", "5",            # Sleep 5s between downloads
        "--max-sleep-interval", "15",       # Randomize up to 15s
        "--limit-rate", "5M",               # Throttle to 5MB/s (mimics streaming)
        "-o", audio_file.replace('.mp3', ''), 
        video_url
    ], capture_output=True, text=True, check=True)

    meta = json.loads(result.stdout)

    # Parse metadata fields
    raw_date = meta.get("upload_date", "")
    historical_date = (
        datetime.strptime(raw_date, "%Y%m%d").strftime("%Y-%m-%d")
        if raw_date else ""
    )
    location    = meta.get("location") or ""
    tags        = meta.get("tags") or []
    description = meta.get("description") or ""
    title       = meta.get("title") or ""

    print("Task Complete")
    print(f"  Title : {title}")
    print(f"  Date  : {historical_date}")
    print(f"  Saved : {audio_file}")

    # ──────────────────────────────────────────────────────────────────────────────
    # 3. Transcribe with Whisper
    if not os.path.exists(audio_file):
        raise FileNotFoundError(f"Audio file '{audio_file}' not found.")

    print("Start transcription...")
    subprocess.run([
        "whisper",
        "--language",        lang_code,
        "--word_timestamps", "True",
        "--model",           model_name,
        "--output_dir",      output_dir,
        "--device",          "cuda",
        audio_file
    ], check=True)

    # ──────────────────────────────────────────────────────────────────────────────
    # 4. Read the plain-text transcript (.txt produced by Whisper)
    #transcript_path = os.path.join(output_dir, os.path.splitext(audio_file)[0] + ".txt")

    print("Transcribed")
    audio_path = Path(audio_file)
    transcript_path = Path(output_dir) / f"{audio_path.stem}.txt"

    # If you need it as a string for the open() function:
    transcript_path = str(transcript_path)
    if not os.path.exists(transcript_path):
        raise FileNotFoundError(
            f"Transcript not found at '{transcript_path}'. "
            "Check the output_dir and audio filename."
        )

    with open(transcript_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # ──────────────────────────────────────────────────────────────────────────────
    # 5. Write CSV
    print("Writing into csv...")
    audio_path_str = str(audio_path)

    row = {
        "politician":      f"{politician}",
        "historical_date": datetime.strptime(historical_date, "%Y-%m-%d").date() if historical_date else "",
        "location":        location,
        "tags":            tags,
        "description":     description.replace("\n", " ").replace("\r", " ").strip(),
        "title":           title.replace("\n", " ").replace("\r", " ").strip(),
        "url":             url,
        "audio_file":      audio_path_str[audio_path_str.index("crossdem"):],
        "text":            text.replace("\n", " ").replace("\r", " ").strip(),
    }

    write_header = not os.path.exists(csv_output)
    with open(csv_output, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    print(f"Done! Row appended to '{csv_output}'")

In [ ]:
# ── Run ─────────────────────────────────────────────────────────────────────
urls = [
    "https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823",
    "https://www.radioradicale.it/scheda/3913/crisi-di-governo-le-consultazioni-di-fanfani?i=2809940",
    "https://www.radioradicale.it/scheda/115568/le-elezioni-anticipate-della-primavera-87-il-governo-fanfani-lopposizione-radicale?i=1806617",
    "https://www.radioradicale.it/scheda/20978/crisi-di-governo-il-presidente-incaricato?i=2725312",
    "https://www.radioradicale.it/scheda/58768/sindacato-crisi-della-sua-rappresentanza-e-sfida-del-cambiamento?i=2419711"
]

for url in urls:
    r = extract_date_and_timestamps(url, speaker="Fanfani")
    for key, value in r.items():
        print(f"{key:<18} {value}")